# Taller: Métricas de evaluación y riesgos ocultos en modelos financieros
## Parte 1: Detección de fraude (Matriz de confusión)

In [ ]:
import numpy as np
import pandas as pd

TP, FN, FP, TN = 10, 40, 20, 930
total = TP + FN + FP + TN

accuracy    = (TP + TN) / total
precision   = TP / (TP + FP)
recall      = TP / (TP + FN)
specificity = TN / (TN + FP)
f1          = 2 * precision * recall / (precision + recall)
error_rate  = (FP + FN) / total

pd.DataFrame({
    "Fórmula": ["(TP+TN)/N", "TP/(TP+FP)", "TP/(TP+FN)", "TN/(TN+FP)",
                "2·P·R/(P+R)", "(FP+FN)/N"],
    "Valor": [accuracy, precision, recall, specificity, f1, error_rate],
}, index=["Accuracy", "Precision", "Recall", "Specificity", "F1 Score", "Error Rate"]).round(4)

### Resultados
| Métrica | Cálculo | Valor |
|---|---|---|
| Accuracy | (10+930)/1000 | **0.94 (94 %)** |
| Precision | 10/(10+20) | **0.3333 (33.3 %)** |
| Recall | 10/(10+40) | **0.20 (20 %)** |
| Specificity | 930/(930+20) | **0.9789 (97.9 %)** |
| F1 Score | 2·0.3333·0.20/(0.3333+0.20) | **0.25** |
| Error Rate | (20+40)/1000 | **0.06 (6 %)** |

### Análisis de riesgo
**La Exactitud engaña por el desbalance de clases.** Sólo 50 de 1,000 transacciones (5 %) son fraude. Un modelo "tonto" que declarara *todo* como legítimo tendría Accuracy = 950/1000 = 95 %, ¡mejor que este modelo (94 %)! La Accuracy está dominada por los Verdaderos Negativos (930) y oculta el desempeño en la clase que importa.

**El Recall revela el fracaso:** Recall = 20 %. El modelo sólo detecta 10 de 50 fraudes; **40 fraudes (80 %) pasan como transacciones legítimas (Falsos Negativos)**. En fraude, un Falso Negativo es dinero perdido de forma directa (contracargos, reembolsos, pérdida de confianza del cliente, sanciones regulatorias). Un Falso Positivo, en cambio, sólo cuesta una llamada al cliente o un bloqueo temporal de tarjeta. Los costos son muy asimétricos, así que una métrica que los pondera igual (Accuracy) es inadecuada.

Además la **Precision es 33 %**: de cada 3 alertas, 2 son falsas, así que el modelo tampoco es eficiente operativamente. El F1 = 0.25 resume ambos problemas.

**Ilustración monetaria** (supuestos editables):

In [ ]:
costo_FN = 500   # pérdida media por fraude no detectado (USD) - supuesto
costo_FP = 10    # costo de revisar/bloquear una transacción legítima (USD) - supuesto

costo_modelo   = FN * costo_FN + FP * costo_FP
costo_sin_modelo = (TP + FN) * costo_FN          # sin modelo, todo fraude pasa
print(f"Pérdida por FN del modelo : ${FN*costo_FN:,}")
print(f"Costo por FP del modelo   : ${FP*costo_FP:,}")
print(f"Costo total con modelo    : ${costo_modelo:,}")
print(f"Costo sin modelo          : ${costo_sin_modelo:,}")
print(f"Fraude evitado            : {1-costo_modelo/costo_sin_modelo:.1%}")

**Conclusión:** aunque Accuracy = 94 %, el modelo deja pasar 4 de cada 5 fraudes. Para el banco es un fracaso financiero: no debe ir a producción hasta mejorar el Recall (p. ej. bajando el umbral de decisión, reponderando clases o usando técnicas para datos desbalanceados) y evaluarse con Recall, Precision-Recall/F1 y costo esperado.

---
## Parte 2: Anatomía del error (Regresión y sesgo)

In [ ]:
df = pd.DataFrame({"Día": [1, 2, 3, 4],
                   "Precio real (y)": [100, 110, 120, 130],
                   "Predicción (ŷ)":  [80, 90, 100, 110]})
df["Error (y-ŷ)"] = df["Precio real (y)"] - df["Predicción (ŷ)"]
df["Error cuadrado"] = df["Error (y-ŷ)"] ** 2
display(df)

mse  = df["Error cuadrado"].mean()
rmse = np.sqrt(mse)
sesgo = df["Error (y-ŷ)"].mean()
var_err = df["Error (y-ŷ)"].var(ddof=0)
print(f"MSE  = {mse:.2f}")
print(f"RMSE = {rmse:.2f}")
print(f"Sesgo (error medio) = {sesgo:.2f}")
print(f"Varianza del error  = {var_err:.2f}")
print(f"Comprobación: MSE = sesgo² + varianza = {sesgo**2 + var_err:.2f}")

### Resultados
| Día | y | ŷ | Error | Error² |
|---|---|---|---|---|
| 1 | 100 | 80 | 20 | 400 |
| 2 | 110 | 90 | 20 | 400 |
| 3 | 120 | 100 | 20 | 400 |
| 4 | 130 | 110 | 20 | 400 |

**MSE = 1600/4 = 400  ·  RMSE = √400 = $20**

### Análisis de riesgo
**Diagnóstico: Alto Sesgo (Underfitting), no Alta Varianza.**

El error es **exactamente +$20 todos los días**: el modelo *subestima sistemáticamente* el precio en la misma magnitud y dirección. Descomposición del MSE sobre estos errores: sesgo² = 20² = 400 y varianza del error = 0, por lo que MSE = 400 + 0. Todo el error es sesgo. Si hubiera alta varianza (overfitting), los errores serían erráticos, con signos y magnitudes cambiantes según la muestra, y el modelo sería inestable ante pequeños cambios en los datos. Aquí la dispersión del error es nula: el modelo es muy estable pero consistentemente equivocado.

**Por qué es un "error de aproximación":** el sesgo mide la distancia entre el promedio de las predicciones del modelo y la verdadera función objetivo. Aquí la familia de funciones del modelo (p. ej. una recta con el intercepto mal ubicado o demasiado simple) no puede aproximar la relación real del precio: capta la tendencia (+$10 por día, pendiente correcta) pero está desplazado $20 hacia abajo. No es ruido aleatorio ni sobreajuste a la muestra; es una limitación estructural de la hipótesis (subajuste) que **no desaparece con más datos**, sólo con un modelo más expresivo o corrigiendo el intercepto/variables omitidas.

**Riesgo financiero:** un sesgo constante es peligroso porque el analista subestimaría siempre el valor del activo (mala valuación, señales de compra/venta erróneas, cobertura insuficiente), y al ser sistemático no se compensa al promediar. La buena noticia: al ser constante es fácil de corregir (ŷ + 20 daría error 0).